# SQLAlchemy
TL;DR:
is python library with
- Python-SQL toolkit (SQLAlchemy core):
    - connection + CRUD (create_read_update_delete) from python to SQL

- SQLAlchemy ORM : object relational mapping :
    - CRUD of databases from python-code with no SQL queries/syntax, only python objects


# SQLAlchemy - core

In [1]:
import pandas as pd
import numpy as np

#import pyodbc ## python -odbc : connection module for T-SQL with python (="driver")
#import psycopg ## connexion postgre

from sqlalchemy import (create_engine ## create a connection
                        ,text ## translate string/text into SQL-query via sqlalchemy
                                ### -> the library allows to pass other objects aswell: see ORM-part of SQLAlchemy
                        )


# connection to T-SQL server + database

In [2]:
### ====== VARIABLES ====== ###
dbms = 'mssql+pyodbc'
driver = 'ODBC+driver+17+for+SQL+Server'
server = 'CEDRIC-VO' 
database = 'DBSlide_OK' 

username = "username"
password = "password"

if False:
    ### !!!! to replace by your credentials
    username = input("user: ")
    ### !!!! to replace by your credentials
    password = input("password: ")
### ====== END VARIABLES ====== ###


### SPECIFIC connnection string for MICROSOFT-SQL SERVER
### other connection_string templates : https://docs.sqlalchemy.org/en/20/core/connections.html

con_string_trusted_windows_connection = (
    f'{dbms}://{server}/{database}?trusted_connection=yes&driver={driver}'
)

### alternative
con_string_sql_auth = (
    f'{dbms}://{server}/{database}?UID={username}&PWD={password}&driver={driver}'
)
### fully configured connection
engine = create_engine(con_string_trusted_windows_connection) 

### postgres SQL


In [3]:
dbms = "postgresql+psycopg"
server = "localhost" 
port = 5432
database = "staging"
username = "username"
password = "password"


con_string_sql_auth_postgre = f'{dbms}://{username}:{password}@{server}:{port}/{database}'
# engine  = create_engine(con_string_sql_auth_postgre)

## testing the connection

In [4]:
open_connection =  engine.connect()

result_query = open_connection.execute(text("""SELECT 'Hello World'""")) 
### result_query = reference/temporary value
### "text"-function required for SQL alchemy: when feeding text-based queries, alternatives= SQLAlchemy-object 
res = (list(result_query)).copy() ## independent copy

open_connection.close() ### need to close the file/connection when done
print (list(result_query)) ### connection closed, now referenced to empty value
print (res) ### copied values, stayed

[]
[('Hello World',)]


In [5]:
with engine.connect() as myconn: ## PYTHON: automatic closing of files and connections via "with .. as .. :"
    result_query = myconn.execute(text("""SELECT 'Hello World 2'"""))
    res = (list(result_query)).copy()
    
print (res)

[('Hello World 2',)]


## no pandas : manual create,read,update,delete

In [6]:
### valid SQL syntax
create_table =  """
                    CREATE TABLE my_python_table
                    (
                        id INT IDENTITY PRIMARY KEY,
                        val VARCHAR(50),
                    )
                """
insert_query =  """
                    INSERT INTO my_python_table (val) VALUES
                     ('a')
                    ,('b')
                    ,('c')
                """

select_query =  """
                    SELECT * FROM my_python_table
                """

drop_table =    """
                    DROP TABLE my_python_table
                """

In [7]:

with engine.connect() as myconn: 
    try:
        myconn.execute(text(drop_table))
    except:
        pass
    
    
    ### create table
    myconn.execute(text(create_table)) 
    
    ### bunch of inserts
    myconn.execute(text(insert_query))
    myconn.execute(text(insert_query))
    
    ### select data
    result_query = myconn.execute(text(select_query))
    res = (list(result_query)).copy()

    ### finalize transaction (default=rollback)
    myconn.commit() 
print('-'*50)
print (res)

--------------------------------------------------
[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'a'), (5, 'b'), (6, 'c')]


In [8]:
list_of_dictionaries = [ {"col":"a"} # col=column name
                        ,{'col':'z'}
                        ,{'col':'e'}
                        ,{'col':'r'}
                        ,{'col':'t'}
                        ,{"col":"y","unused_column":np.nan}] 
    ### can be list,tupple;
            


block_insert =  """
                    INSERT INTO my_python_table (val) VALUES 
                    (:col
                    )
                """ 
### ":col" the result/value from "dictionary[col]"



with engine.connect() as myconn: 
    
    
    ### block insert
    myconn.execute(text(block_insert),list_of_dictionaries)
    
    ### select data
    result_query = myconn.execute(text(select_query))
    res = (list(result_query)).copy()

    ### drop table
    myconn.execute(text(drop_table)) 

    ### finalize transaction (default=rollback)
    #myconn.rollback()
    
print('-'*50)
print (res)

--------------------------------------------------
[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'a'), (5, 'b'), (6, 'c'), (7, 'a'), (8, 'z'), (9, 'e'), (10, 'r'), (11, 't'), (12, 'y')]


In [9]:
### previous insert and drop tables have been undone when rollback :

with engine.connect() as myconn:
    ### select data
    result_query = myconn.execute(text(select_query))
    res = (list(result_query)).copy()
print(res) 

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'a'), (5, 'b'), (6, 'c')]


## with pandas 

In [10]:
df = pd.DataFrame(data={"id":[1,2,3],"val":list('aba')})

## possible with engine.connect()
with engine.connect() as con:
    ### create/replace table
    df.to_sql(name='my_df',con=con,if_exists='replace',index=True) ## autocommit=true

    ### read table
    df_from_sql = pd.read_sql_table(table_name='my_df',con=con)

    ### read query
    df_from_sql_2 = pd.read_sql_query(sql="SELECT * FROM my_df",con=con)


## without "with engine.connect()" : directly with engine
df.to_sql(name='my_df',con=engine,if_exists='replace',index=True) ## autocommit=true

### read table
df_from_sql_3 = pd.read_sql_table(table_name='my_df',con=engine)

### read query
df_from_sql_4 = pd.read_sql_query(sql="SELECT * FROM my_df",con=engine)
   
display(df_from_sql.head(3))
display(df_from_sql_2.head(3))
display(df_from_sql_3.head(3))
display(df_from_sql_4.head(3))

,index,id,val
0,0,1,a
1,1,2,b
2,2,3,a


,index,id,val
0,0,1,a
1,1,2,b
2,2,3,a


,index,id,val
0,0,1,a
1,1,2,b
2,2,3,a


,index,id,val
0,0,1,a
1,1,2,b
2,2,3,a


In [11]:
with engine.connect() as con:
### BAD PRACTICE: DON'T DO THIS !!!
    try:
        pd.read_sql_query(sql="DROP TABLE my_df",con=con) 
        ### executed but return error: use sql alchemy to modify , NOT PANDAS !
        ### this query doesn't return a row (no select)
    except Exception as e:
        print(e)

    try:
        df_from_sql = pd.read_sql_table(table_name='my_df',con=con)
        ### table dropped from above
    except Exception as e:
        print(e)


This result object does not return rows. It has been closed automatically.
Table my_df not found
